# TRAIN EyePACS v2 — all seeds, ViT-Base stabilized

Supersedes `TRAIN_eyepacs_v2_SEEDS.ipynb`. Covers replicates 0, 1 and 2 in one place.
Replicate 0 is the canonical run and keeps its original, untagged file names;
replicates 1 and 2 carry `_seed1` / `_seed2`. Anything already checkpointed is skipped.

**The one change: ViT-Base now uses the stabilized ViT recipe** established earlier in this
project (lr 3e-5, 5-epoch linear warmup then cosine, layer-wise LR decay 0.65 per block,
weight decay 0.05, gradient clipping 1.0). v1's config gave ViT-Base CNN settings (lr 1e-4,
no warmup), known to be unstable for ViT fine-tuning; seed 1 confirmed it (val AUC 0.61 at
epoch 1, 0.75 at epoch 7). Applied to both ViT-Base variants in every replicate.

Unchanged: CNNs (v1 settings), RETFound (low LR, LLRD, clipping, weight decay; trained
stably at 15 epochs, so no warmup added), 15 epochs throughout, best-val-AUC checkpointing.

**Before running:** the old ViT-Base checkpoints and predictions must be cleared so they are
retrained. Claude does this; confirm it is done first.

Prerequisites: A100 runtime; Colab secret `HF_TOKEN`. Run cells 1-5.

In [1]:
# Cell 1: auth, deps, Drive helpers, RETFound preflight
!pip -q install timm transformers
import io, os, json, time, zipfile, glob, datetime, random, math
import numpy as np, pandas as pd, torch
import torch.nn as nn
from google.colab import auth, userdata
auth.authenticate_user()
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')

F_SPLITS      = '13x_CS7N_UasGRSuNYMyD5To2RTYjL-EP'
F_CHECKPOINTS = '1DAAXJ2fjk2zAzx7PpGaWsk4BXudeA5Le'
F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_AUDIT       = '1zuvfcxC3umHAq_uf7Vx9omMEgUEuQhTt'
SPLIT_IDS = {'train': '1FU30c6vRYMLqX8h9z2F58iEqow1Zx8NQ',
             'val':   '1Z9tFlAGe0j4RVRgG9-6LnhTrukvFy87y',
             'test':  '1mSlAqeNtFJ0Ap6BMenrrQxHaWep-9nEj'}
IMAGE_ZIPS = {'batch1': '11WspNNHS73OadbW-sqy7BSzcrsBkX1d6',
              'batch2': '1zG7rHjm8Ty9LHG4LqMzgnOzrT_90VO9H',
              'batch3': '1L88vUNSxWjdGPBquhknZhafcnXXRd3ES',
              'batch4': '1-FrpGB74qEGzDvXlA9FhmB2OeeZkKPAN',
              'batch5': '1mEIrSGVExZ6zmxnRLxyFab5yB1kH5YBT'}
RETFOUND_REPO = 'iszt/RETFound_mae_meh'
NUM_WORKERS = 8

def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=50*1024*1024); done = False
        while not done: _, done = d.next_chunk()

def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false",
                         fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None

def up(path, folder, mime='application/octet-stream', overwrite=False):
    media = MediaFileUpload(path, mimetype=mime, resumable=True); name = os.path.basename(path)
    if overwrite:
        fid = find_id(name, folder)
        if fid: svc.files().update(fileId=fid, media_body=media).execute(); return
    svc.files().create(body={'name': name, 'parents': [folder]}, media_body=media, fields='id').execute()
    print('  uploaded', name)

def existing(folder):
    names, tok = set(), None
    while True:
        r = svc.files().list(q=f"'{folder}' in parents and trashed=false",
                             fields='files(name),nextPageToken', pageToken=tok, pageSize=1000).execute()
        names |= {f['name'] for f in r.get('files', [])}; tok = r.get('nextPageToken')
        if not tok: break
    return names

STATUS = {'started': datetime.datetime.now(datetime.timezone.utc).isoformat(),
          'run': 'all seeds, ViT-Base stabilized', 'events': []}
def heartbeat(**kw):
    STATUS.update(kw); STATUS['updated'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with open('TRAINING_STATUS.json', 'w') as f: json.dump(STATUS, f, indent=2)
    try: up('TRAINING_STATUS.json', F_AUDIT, 'application/json', overwrite=True)
    except Exception as e: print('  (heartbeat failed, continuing):', e)

def find_blocks(m):
    best = None
    for _, mod in m.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) >= 12 and (best is None or len(mod) > len(best)):
            best = mod
    if best is None: raise RuntimeError('could not locate transformer blocks')
    return best

from transformers import AutoModel, AutoConfig
_m = AutoModel.from_pretrained(RETFOUND_REPO, add_pooling_layer=False)
_n = sum(p.numel() for p in _m.parameters())
assert _n > 2.9e8 and len(find_blocks(_m)) == 24, 'RETFound preflight failed'
print(f'GPU: {torch.cuda.get_device_name(0)} | RETFound preflight passed ({_n/1e6:.1f}M)'); del _m

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

GPU: NVIDIA A100-SXM4-40GB | RETFound preflight passed (303.3M)


In [2]:
# Cell 2: roster. ViT-Base uses the stabilized recipe; everything else unchanged.
VIT_STABLE = {'lr': 3e-5, 'wd': 0.05, 'clip': 1.0, 'lldecay': 0.65, 'warmup': 5, 'recipe': 'vit_stable'}
MODELS = [
  {'name':'densenet121_binary','arch':'densenet121','task':'binary','nc':1,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'densenet121_5class','arch':'densenet121','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'efficientnet_b3_binary','arch':'efficientnet_b3','task':'binary','nc':1,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'efficientnet_b3_5class','arch':'efficientnet_b3','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'resnet50_binary','arch':'resnet50','task':'binary','nc':1,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'resnet50_5class','arch':'resnet50','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'vit_base_binary','arch':'vit_base_patch16_224','task':'binary','nc':1,'epochs':15,'bs':32, **VIT_STABLE},
  {'name':'vit_base_5class','arch':'vit_base_patch16_224','task':'5class','nc':5,'epochs':15,'bs':32, **VIT_STABLE},
  {'name':'retfound_binary','arch':'retfound','task':'binary','nc':1,'epochs':15,'bs':8,'lr':5e-5,'wd':0.05,'clip':1.0,'lldecay':0.65},
  {'name':'retfound_5class','arch':'retfound','task':'5class','nc':5,'epochs':15,'bs':8,'lr':5e-5,'wd':0.05,'clip':1.0,'lldecay':0.65},
  {'name':'retfound_adversarial','arch':'retfound','task':'adversarial_binary','nc':1,'epochs':15,'bs':8,'lr':3e-5,'wd':0.05,'clip':1.0,'lldecay':0.65,'alpha':0.08,'primary':'densenet121_binary'},
]
ALL = [m['name'] for m in MODELS]
REPLICATES = [0, 1, 2]
BASE_SEED = 20260920
IMG_SIZE = 224
def tag(k): return '' if k == 0 else f'_seed{k}'     # replicate 0 keeps canonical names
print('replicates:', REPLICATES, '| ViT-Base recipe:', VIT_STABLE)

replicates: [0, 1, 2] | ViT-Base recipe: {'lr': 3e-05, 'wd': 0.05, 'clip': 1.0, 'lldecay': 0.65, 'warmup': 5, 'recipe': 'vit_stable'}


In [3]:
# Cell 3: fetch splits and images, verify coverage
os.makedirs('/content/img', exist_ok=True)
splits = {}
for s, fid in SPLIT_IDS.items():
    p = f'split_{s}.csv'
    if not os.path.exists(p): dl(fid, p)
    splits[s] = pd.read_csv(p)
heartbeat(stage='downloading images')
if not glob.glob('/content/img/**/*.jp*g', recursive=True):
    for name, fid in IMAGE_ZIPS.items():
        z = f'/content/{name}.zip'
        if not os.path.exists(z): print('downloading', name); dl(fid, z)
        with zipfile.ZipFile(z) as zf: zf.extractall('/content/img')
        os.remove(z); heartbeat(stage=f'extracted {name}')
paths = glob.glob('/content/img/**/*.jp*g', recursive=True) + glob.glob('/content/img/**/*.png', recursive=True)
INDEX = {os.path.splitext(os.path.basename(p))[0]: p for p in paths}
need = set().union(*[set(d['image_id'].astype(str)) for d in splits.values()])
missing = sorted(need - set(INDEX))
print('images on disk:', len(INDEX), '| missing:', len(missing))
if missing: raise SystemExit('Images missing. Stop and report.')

downloading batch1
downloading batch2
downloading batch3
downloading batch4
downloading batch5
images on disk: 19332 | missing: 0


In [4]:
# Cell 4: dataset, builders, optimizers (incl. ViT LLRD), warmup scheduler, train, predict
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
TF_TRAIN = transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2,0.2,0.1), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DR(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(INDEX[str(r['image_id'])]).convert('RGB')), int(r['level']), int(r['binary_label']), i

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class RETFound(nn.Module):
    def __init__(self, nc):
        super().__init__()
        self.base = AutoModel.from_pretrained(RETFOUND_REPO, add_pooling_layer=False)
        h = self.base.config.hidden_size
        self.fc_norm = nn.LayerNorm(h, eps=1e-6); self.classifier = nn.Linear(h, nc)
        nn.init.trunc_normal_(self.classifier.weight, std=0.01); nn.init.zeros_(self.classifier.bias)
    def forward(self, x):
        h = self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:, 1:].mean(dim=1)))

def build(cfg):
    if cfg['arch'] == 'retfound':
        m = RETFound(cfg['nc'])
        if sum(p.numel() for p in m.base.parameters()) < 2.9e8: raise RuntimeError('RETFound weights did not load')
        return m
    return timm.create_model(cfg['arch'], pretrained=True, num_classes=cfg['nc'])

def _llrd_groups(named, blocks, lr, d, embed_prefixes, head_prefixes, head_mult):
    n = len(blocks)
    bid = {id(p) for b in blocks for p in b.parameters()}
    embed = [p for nm, p in named if id(p) not in bid and nm.startswith(embed_prefixes)]
    head = [p for nm, p in named if id(p) not in bid and nm.startswith(head_prefixes)]
    used = bid | {id(p) for p in embed} | {id(p) for p in head}
    rest = [p for nm, p in named if id(p) not in used]
    g = [{'params': embed, 'lr': lr * d ** n}]
    g += [{'params': list(b.parameters()), 'lr': lr * d ** (n - 1 - i)} for i, b in enumerate(blocks)]
    g += [{'params': rest, 'lr': lr}, {'params': head, 'lr': lr * head_mult}]
    return [x for x in g if x['params']]

def make_optimizer(model, cfg):
    lr, wd = cfg['lr'], cfg['wd']
    if cfg['arch'] == 'retfound':
        named = [('base.' + nm, p) for nm, p in model.base.named_parameters()] + \
                [('fc_norm.' + nm, p) for nm, p in model.fc_norm.named_parameters()] + \
                [('classifier.' + nm, p) for nm, p in model.classifier.named_parameters()]
        embed_pref = tuple('base.' + x for x in ('embeddings',)) + ('base.embeddings',)
        groups = _llrd_groups(named, find_blocks(model.base), lr, cfg['lldecay'],
                              ('base.embeddings',), ('classifier',), 10)
        return torch.optim.AdamW(groups, lr=lr, weight_decay=wd)
    if cfg.get('recipe') == 'vit_stable':
        groups = _llrd_groups(list(model.named_parameters()), find_blocks(model), lr, cfg['lldecay'],
                              ('cls_token', 'pos_embed', 'patch_embed', 'reg_token'), ('head',), 1)
        print(f'  ViT LLRD {cfg["lldecay"]}: {len(groups)} groups, head lr {lr:.1e}')
        return torch.optim.AdamW(groups, lr=lr, weight_decay=wd)
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

def make_scheduler(opt, cfg):
    E, W = cfg['epochs'], cfg.get('warmup', 0)
    if W:
        f = lambda ep: (ep + 1) / W if ep < W else 0.5 * (1 + math.cos(math.pi * (ep - W) / max(1, E - W)))
        return torch.optim.lr_scheduler.LambdaLR(opt, f)
    return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E)

@torch.no_grad()
def predict(model, df, dev, bs=64):
    model.eval()
    dl_ = DataLoader(DR(df, TF_EVAL), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    probs = np.zeros(len(df), dtype=np.float64)
    for x, _, _, idx in dl_:
        out = model(x.to(dev, non_blocking=True)).float()
        p = torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
        probs[idx.numpy()] = p.double().cpu().numpy()
    return probs

def train_one(cfg, k, dev, seed, eval_bs, label):
    seed_everything(seed)
    tr, va = splits['train'], splits['val']
    model = build(cfg).to(dev); primary_err = None
    if cfg['task'] == 'adversarial_binary':
        pf = f"eyepacs_predictions_train_{cfg['primary']}{tag(k)}_v2.csv"
        if not os.path.exists(pf):
            fid = find_id(pf, F_PREDICTIONS)
            if not fid: raise RuntimeError(f'need {pf} first')
            dl(fid, pf)
        pp = pd.read_csv(pf).set_index('image_id')['predicted_probability']
        primary_err = torch.tensor((pp.reindex(tr['image_id'].astype(str)).values >= 0.5).astype(float)
                                   - tr['binary_label'].values, dtype=torch.float32)
    g = torch.Generator(); g.manual_seed(seed)
    dl_ = DataLoader(DR(tr, TF_TRAIN), batch_size=cfg['bs'], shuffle=True, num_workers=NUM_WORKERS,
                     drop_last=True, generator=g, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    opt = make_optimizer(model, cfg); sch = make_scheduler(opt, cfg)
    lossf = nn.BCEWithLogitsLoss() if cfg['nc'] == 1 else nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')
    best_auc, best_state, best_ep = -1.0, None, -1
    for ep in range(cfg['epochs']):
        model.train(); tot = 0.0; t0 = time.time()
        for x, lvl, lab, idx in dl_:
            x = x.to(dev, non_blocking=True)
            y = (lab.float() if cfg['nc'] == 1 else lvl.long()).to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                out = model(x)
                loss = lossf(out[:, 0], y) if cfg['nc'] == 1 else lossf(out, y)
                if primary_err is not None:
                    es = torch.sigmoid(out[:, 0].float()) - lab.float().to(dev); pe = primary_err[idx].to(dev)
                    if es.std() > 1e-6 and pe.std() > 1e-6:
                        loss = loss + cfg['alpha'] * ((es - es.mean()) * (pe - pe.mean())).mean() / (es.std() * pe.std())
            scaler.scale(loss).backward()
            if cfg.get('clip'): scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            scaler.step(opt); scaler.update(); tot += float(loss.item())
        sch.step()
        vp = predict(model, va, dev, eval_bs); vauc = float(roc_auc_score(va['binary_label'].values, vp))
        star = ''
        if vauc > best_auc:
            best_auc, best_ep = vauc, ep + 1
            best_state = {k_: v.detach().cpu().clone() for k_, v in model.state_dict().items()}; star = '  <- best'
        lr_now = opt.param_groups[-1]['lr']
        print(f"  epoch {ep+1}/{cfg['epochs']}  loss {tot/max(1,len(dl_)):.4f}  val_auc {vauc:.4f}  lr {lr_now:.2e}  ({time.time()-t0:.0f}s){star}")
        heartbeat(stage='training', current=label, epoch=f"{ep+1}/{cfg['epochs']}",
                  last_val_auc=round(vauc, 4), best_val_auc=round(best_auc, 4), best_epoch=best_ep)
    model.load_state_dict(best_state)
    return model, best_auc, best_ep

In [6]:
# Cell 5: run every (replicate, model) not yet checkpointed. Replicate 0 first.
dev = 'cuda'
have = existing(F_CHECKPOINTS)
todo = [(k, m) for k in REPLICATES for m in MODELS if f"eyepacs_{m['name']}{tag(k)}_v2.pt" not in have]
heartbeat(stage='starting', queued=[f"{m['name']}{tag(k) or '_seed0'}" for k, m in todo])
print(f'to train this session: {len(todo)} of {len(REPLICATES) * len(ALL)}')
for k, m in todo: print('  ', m['name'] + (tag(k) or ' (seed 0)'))

for i, (k, cfg) in enumerate(todo, 1):
    seed = BASE_SEED + 1000 * k + ALL.index(cfg['name'])        # k=0 reproduces the canonical seeds
    eval_bs = 32 if cfg['arch'] == 'retfound' else 64
    label = cfg['name'] + (tag(k) or '_seed0')
    print(f'\n=== [{i}/{len(todo)}] {label} (seed {seed}) ===')
    t0 = time.time()
    try:
        model, best_auc, best_ep = train_one(cfg, k, dev, seed, eval_bs, label)
    except Exception as e:
        print('  FAILED:', e); STATUS['events'].append({'model': label, 'status': 'failed', 'error': str(e)})
        heartbeat(stage='model failed', current=label); continue
    ck = f"eyepacs_{cfg['name']}{tag(k)}_v2.pt"
    torch.save({'state_dict': model.state_dict(), 'config': cfg, 'seed': seed, 'replicate': k,
                'best_val_auc': best_auc, 'best_epoch': best_ep}, ck)
    up(ck, F_CHECKPOINTS)
    for s in ['train', 'val', 'test']:
        d = splits[s]; pr = predict(model, d, dev, eval_bs)
        fn = f"eyepacs_predictions_{s}_{cfg['name']}{tag(k)}_v2.csv"
        pd.DataFrame({'image_id': d['image_id'].astype(str), 'patient_id': d['patient_id'].astype(str),
                      'eye': d['eye'], 'dataset': 'eyepacs', 'split': s,
                      'true_grade': d['level'].astype(int), 'true_label': d['binary_label'].astype(int),
                      'predicted_probability': pr, 'predicted_class': (pr >= 0.5).astype(int),
                      'model_name': cfg['name'], 'replicate': k}).to_csv(fn, index=False, float_format='%.17g')
        up(fn, F_PREDICTIONS, 'text/csv')
    mins = round((time.time() - t0) / 60, 1)
    STATUS['events'].append({'model': label, 'status': 'ok', 'minutes': mins,
                             'best_val_auc': round(best_auc, 4), 'best_epoch': best_ep, 'seed': seed})
    heartbeat(stage='model complete', current=label)
    print(f'  done in {mins} min (best val_auc {best_auc:.4f} @ epoch {best_ep})')
    del model; torch.cuda.empty_cache()

done = existing(F_CHECKPOINTS)
left = [m + (tag(k) or '_seed0') for k in REPLICATES for m in ALL if f'eyepacs_{m}{tag(k)}_v2.pt' not in done]
heartbeat(stage='session finished', remaining=left)
print(f'\nremaining: {len(left)}', left)

to train this session: 18 of 33
   vit_base_binary (seed 0)
   vit_base_5class (seed 0)
   vit_base_binary_seed1
   vit_base_5class_seed1
   retfound_binary_seed1
   retfound_5class_seed1
   retfound_adversarial_seed1
   densenet121_binary_seed2
   densenet121_5class_seed2
   efficientnet_b3_binary_seed2
   efficientnet_b3_5class_seed2
   resnet50_binary_seed2
   resnet50_5class_seed2
   vit_base_binary_seed2
   vit_base_5class_seed2
   retfound_binary_seed2
   retfound_5class_seed2
   retfound_adversarial_seed2

=== [1/18] vit_base_binary_seed0 (seed 20260926) ===
  FAILED: could not locate transformer blocks

=== [2/18] vit_base_5class_seed0 (seed 20260927) ===
  FAILED: could not locate transformer blocks

=== [3/18] vit_base_binary_seed1 (seed 20261926) ===
  FAILED: could not locate transformer blocks

=== [4/18] vit_base_5class_seed1 (seed 20261927) ===
  FAILED: could not locate transformer blocks

=== [5/18] retfound_binary_seed1 (seed 20261928) ===


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  epoch 1/15  loss 0.4151  val_auc 0.8374  lr 4.95e-04  (175s)  <- best
  epoch 2/15  loss 0.3575  val_auc 0.8661  lr 4.78e-04  (173s)  <- best
  epoch 3/15  loss 0.3449  val_auc 0.8605  lr 4.52e-04  (172s)
  epoch 4/15  loss 0.3386  val_auc 0.8778  lr 4.17e-04  (173s)  <- best
  epoch 5/15  loss 0.3330  val_auc 0.8690  lr 3.75e-04  (172s)
  epoch 6/15  loss 0.3285  val_auc 0.8788  lr 3.27e-04  (172s)  <- best
  epoch 7/15  loss 0.3175  val_auc 0.8756  lr 2.76e-04  (172s)
  epoch 8/15  loss 0.3101  val_auc 0.8684  lr 2.24e-04  (172s)
  epoch 9/15  loss 0.3061  val_auc 0.8750  lr 1.73e-04  (171s)
  epoch 10/15  loss 0.2961  val_auc 0.8787  lr 1.25e-04  (171s)
  epoch 11/15  loss 0.2961  val_auc 0.8767  lr 8.27e-05  (173s)
  epoch 12/15  loss 0.2928  val_auc 0.8754  lr 4.77e-05  (173s)
  epoch 13/15  loss 0.2879  val_auc 0.8746  lr 2.16e-05  (173s)
  epoch 14/15  loss 0.2862  val_auc 0.8754  lr 5.46e-06  (172s)
  epoch 15/15  loss 0.2884  val_auc 0.8756  lr 0.00e+00  (172s)
  uploaded ey

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  epoch 1/15  loss 0.7680  val_auc 0.8485  lr 4.95e-04  (172s)  <- best


...

  epoch 5/15  loss 0.6115  val_auc 0.8720  lr 3.75e-04  (174s)
  epoch 6/15  loss 0.5991  val_auc 0.8789  lr 3.27e-04  (174s)  <- best
  epoch 7/15  loss 0.5787  val_auc 0.8663  lr 2.76e-04  (173s)
  epoch 8/15  loss 0.5708  val_auc 0.8761  lr 2.24e-04  (174s)
  epoch 9/15  loss 0.5588  val_auc 0.8786  lr 1.73e-04  (174s)
  epoch 10/15  loss 0.5451  val_auc 0.8755  lr 1.25e-04  (175s)
  epoch 11/15  loss 0.5384  val_auc 0.8773  lr 8.27e-05  (173s)
  epoch 12/15  loss 0.5277  val_auc 0.8713  lr 4.77e-05  (174s)
  epoch 13/15  loss 0.5242  val_auc 0.8724  lr 2.16e-05  (173s)
  epoch 14/15  loss 0.5209  val_auc 0.8735  lr 5.46e-06  (175s)
  epoch 15/15  loss 0.5134  val_auc 0.8739  lr 0.00e+00  (173s)
  uploaded eyepacs_retfound_5class_seed2_v2.pt
  uploaded eyepacs_predictions_train_retfound_5class_seed2_v2.csv
  uploaded eyepacs_predictions_val_retfound_5class_seed2_v2.csv
  uploaded eyepacs_predictions_test_retfound_5class_seed2_v2.csv
  done in 46.0 min (best val_auc 0.8789 @ epoch 6)

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  epoch 1/15  loss 0.4584  val_auc 0.7932  lr 2.97e-04  (177s)  <- best
  epoch 2/15  loss 0.3957  val_auc 0.8425  lr 2.87e-04  (176s)  <- best
  epoch 3/15  loss 0.3737  val_auc 0.8639  lr 2.71e-04  (176s)  <- best
  epoch 4/15  loss 0.3603  val_auc 0.8684  lr 2.50e-04  (176s)  <- best
  epoch 5/15  loss 0.3548  val_auc 0.8725  lr 2.25e-04  (176s)  <- best
  epoch 6/15  loss 0.3445  val_auc 0.8679  lr 1.96e-04  (175s)
  epoch 7/15  loss 0.3408  val_auc 0.8813  lr 1.66e-04  (176s)  <- best


  epoch 8/15  loss 0.3370  val_auc 0.8752  lr 1.34e-04  (176s)


  epoch 9/15  loss 0.3315  val_auc 0.8786  lr 1.04e-04  (174s)
  epoch 10/15  loss 0.3252  val_auc 0.8789  lr 7.50e-05  (174s)
  epoch 11/15  loss 0.3297  val_auc 0.8784  lr 4.96e-05  (175s)
  epoch 12/15  loss 0.3222  val_auc 0.8787  lr 2.86e-05  (175s)
  epoch 13/15  loss 0.3175  val_auc 0.8774  lr 1.30e-05  (176s)
  epoch 14/15  loss 0.3179  val_auc 0.8779  lr 3.28e-06  (176s)
  epoch 15/15  loss 0.3135  val_auc 0.8783  lr 0.00e+00  (175s)
  uploaded eyepacs_retfound_adversarial_seed2_v2.pt
  uploaded eyepacs_predictions_train_retfound_adversarial_seed2_v2.csv
  uploaded eyepacs_predictions_val_retfound_adversarial_seed2_v2.csv
  uploaded eyepacs_predictions_test_retfound_adversarial_seed2_v2.csv
  done in 46.6 min (best val_auc 0.8813 @ epoch 7)

remaining: 1 ['vit_base_5class_seed2']
